# Tech Challenge - Fase 1: Modelagem Preditiva de NPS
**Objetivo:** Utilizar os dados tratados e validados na Análise Exploratória para treinar um modelo de Classificação (Inteligência Artificial) capaz de prever proativamente quais clientes se tornarão Detratores devido a falhas operacionais.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

## 1. Carregamento dos Dados e Preparação das "Features" (Dummies)
Iniciamos carregando a base já tratada no notebook de exploração. A única transformação adicional necessária é converter a variável categórica de texto (`customer_region`) em colunas binárias (Numéricas) utilizando a técnica de *One-Hot Encoding*, para que o algoritmo consiga interpretá-las.

In [2]:
# Carregando os dados limpos do Notebook 1
df_ml = pd.read_csv('dados_preparados_ml.csv')

# Transformando a região em variáveis Dummies
df_ml = pd.get_dummies(df_ml, columns=['customer_region'], drop_first=True)

# Separando as variáveis preditoras (X) da variável alvo (y)
# Removemos o 'nps_score' numérico para que o modelo preveja apenas a classe
X = df_ml.drop(columns=['nps_score', 'nps_class', 'customer_id', 'order_id'])
y = df_ml['nps_class']

## 2. Separação de Treino e Teste & Treinamento do Modelo
Vamos dividir a nossa base dedicando 80% dos dados para ensinar a máquina a reconhecer os padrões, e preservando 20% para testar o seu aprendizado em cenários que ela nunca viu. Optamos pelo algoritmo **Random Forest Classifier**, ideal para capturar as não linearidades e quebras comportamentais mapeadas na EDA.

In [3]:
# Divisão de 80% para treino e 20% para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instanciando e treinando o algoritmo
classificador = RandomForestClassifier(random_state=42)
classificador.fit(X_train, y_train)

# Realizando as previsões com os dados de teste
y_pred = classificador.predict(X_test)

## 3. Avaliação de Desempenho e Validação Cruzada (Robustez)
O foco central do negócio é a proatividade sobre os **Detratores**. Portanto, a métrica mais importante a ser analisada é o *Recall* (Taxa de Captura) para esta classe específica. Adicionalmente, aplicaremos uma Validação Cruzada (*Cross-Validation*) para assegurar que o modelo não está viciado (*overfitting*).

In [5]:
# Relatório de Métricas (Precision, Recall, F1-Score)
print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(classification_report(y_test, y_pred))

# Teste de Robustez do Modelo
scores = cross_val_score(classificador, X, y, cv=5, scoring='accuracy')

print("\n--- TESTE DE ROBUSTEZ (VALIDAÇÃO CRUZADA - 5 Folds) ---")
print(f"Notas dos testes isolados: {scores.round(3)}")
print(f"Acurácia Média Real: {scores.mean():.3f}")
print(f"Variação (Desvio Padrão): {scores.std():.3f} (Indica alta estabilidade e baixo risco de overfitting)")

--- RELATÓRIO DE CLASSIFICAÇÃO ---
              precision    recall  f1-score   support

    Detrator       0.80      0.99      0.88       363
      Neutro       0.50      0.04      0.08        98
    Promotor       0.95      1.00      0.97        39

    accuracy                           0.80       500
   macro avg       0.75      0.68      0.64       500
weighted avg       0.75      0.80      0.73       500


--- TESTE DE ROBUSTEZ (VALIDAÇÃO CRUZADA - 5 Folds) ---
Notas dos testes isolados: [0.824 0.818 0.802 0.822 0.826]
Acurácia Média Real: 0.818
Variação (Desvio Padrão): 0.009 (Indica alta estabilidade e baixo risco de overfitting)
